Min Max values (bottom-up):

D(MIN) = min(5,7,1) = 1
E(MIN) = min(9,2) = 2
F(MIN) = min(5,10) = 5
G(MIN) = min(12,6) = 6
H(MIN) = 20
B(MAX) = max(1,2) = 2
C(MAX) = max(5,6,20) = 20
A(MIN) = min(2,20) = 2

Alpha-Beta Pruning trace:

Visit I=5, J=7, K=1 → D=1. B gets α=1

Visit L=9, M=2 → E=2. B = max(1,2)=2. B's α=2

A is MIN, β=2. Pass to C: α=-∞, β=2

Visit N=5, O=10 → F=5. C's α=5 > A's β=2 
Prune G and H entirely

A(MIN) = min(2,5) = 2

Pruned nodes: P, Q, R, G, H (entire right subtree of C after F)

In [1]:
import math

def alpha_beta_defense(tree):
    sorted_tree = sorted(tree, key=lambda x: min(x), reverse=True)

    def minimax(node, depth, alpha, beta, is_max_player):
        if isinstance(node, int):
            return node
        
        if is_max_player:
            value = -math.inf
            for child in node:
                value = max(value, minimax(child, depth + 1, alpha, beta, False))
                alpha = max(alpha, value)
                if alpha >= beta:
                    break  
            return value
        else:
            value = math.inf
            for child in node:
                value = min(value, minimax(child, depth + 1, alpha, beta, True))
                beta = min(beta, value)
                if alpha >= beta:
                    break 
            return value

    root_value = minimax(sorted_tree, 0, -math.inf, math.inf, True)
    return root_value

tree_data = [[10, 5, 2], [8, 4, 3], [20, 15, 9]]
result = alpha_beta_defense(tree_data)
print(f"Optimal Defense Score: {result}")

Optimal Defense Score: 9


In [2]:
import math

class SuperTicTacToe:
    def __init__(self):
        self.board = [[' ' for _ in range(9)] for _ in range(9)]
        self.big_board = [' ' for _ in range(9)] 
        self.next_sub_board = -1 

    def check_line(self, line):
        return line[0] != ' ' and line[0] == line[1] == line[2]

    def check_winner(self, b):
        for i in range(0, 9, 3):
            if self.check_line(b[i:i+3]): return b[i]
        for i in range(3):
            if self.check_line([b[i], b[i+3], b[i+6]]): return b[i]
        if self.check_line([b[0], b[4], b[8]]): return b[0]
        if self.check_line([b[2], b[4], b[6]]): return b[2]
        return None

    def evaluate(self):
        score = 0
        res = self.check_winner(self.big_board)
        if res == 'X': return 1000
        if res == 'O': return -1000
        
        for i in range(9):
            if self.big_board[i] == 'X': score += 10
            elif self.big_board[i] == 'O': score -= 10
        return score

    def get_valid_moves(self):
        moves = []
        if self.next_sub_board != -1 and self.big_board[self.next_sub_board] == ' ':
            sub = self.next_sub_board
            for i in range(9):
                if self.board[sub][i] == ' ': moves.append((sub, i))
        else:
            for sub in range(9):
                if self.big_board[sub] == ' ':
                    for i in range(9):
                        if self.board[sub][i] == ' ': moves.append((sub, i))
        return moves

    def alpha_beta(self, depth, alpha, beta, is_max):
        score = self.evaluate()
        if abs(score) == 1000 or depth == 0 or not self.get_valid_moves():
            return score

        if is_max:
            max_eval = -math.inf
            for sub, cell in self.get_valid_moves():
                # Make move
                self.board[sub][cell] = 'X'
                prev_big = self.big_board[sub]
                self.big_board[sub] = self.check_winner(self.board[sub]) or ' '
                prev_next = self.next_sub_board
                self.next_sub_board = cell
                
                eval = self.alpha_beta(depth - 1, alpha, beta, False)
                
                # Undo move
                self.board[sub][cell] = ' '
                self.big_board[sub] = prev_big
                self.next_sub_board = prev_next
                
                max_eval = max(max_eval, eval)
                alpha = max(alpha, eval)
                if beta <= alpha: break
            return max_eval
        else:
            min_eval = math.inf
            for sub, cell in self.get_valid_moves():
                self.board[sub][cell] = 'O'
                prev_big = self.big_board[sub]
                self.big_board[sub] = self.check_winner(self.board[sub]) or ' '
                prev_next = self.next_sub_board
                self.next_sub_board = cell
                
                eval = self.alpha_beta(depth - 1, alpha, beta, True)
                
                self.board[sub][cell] = ' '
                self.big_board[sub] = prev_big
                self.next_sub_board = prev_next
                
                min_eval = min(min_eval, eval)
                beta = min(beta, eval)
                if beta <= alpha: break
            return min_eval

    def get_best_move(self):
        best_val = -math.inf
        best_move = None
        for sub, cell in self.get_valid_moves():
            self.board[sub][cell] = 'X'
            val = self.alpha_beta(4, -math.inf, math.inf, False) 
            self.board[sub][cell] = ' '
            if val > best_val:
                best_val = val
                best_move = (sub, cell)
        return best_move

game = SuperTicTacToe()
print(f"AI Best Move (Sub-board, Cell): {game.get_best_move()}")

AI Best Move (Sub-board, Cell): (0, 0)
